# Retrieval Evaluation (Hit Rate + MRR)

This notebook evaluates vector retrieval quality with a simple and explicit protocol.

Goal:
- Create synthetic questions from each **chunk**.
- Use those questions as ground truth queries.
- Check whether the original source chunk is retrieved.


## Evaluation Design (Simple)

We keep one clear rule for relevance:
- **Relevant item = the source chunk that generated the question**.

Protocol:
1. Load chunks from Chroma collection.
2. Generate 5 synthetic questions per chunk.
3. Store them in a reusable ground-truth file.
4. Retrieve top-k chunks for each question.
5. Compute metrics:

- **Hit Rate@k** = fraction of questions where expected chunk appears in top-k.
- **MRR@k** = average reciprocal rank of the expected chunk (`1/rank` if found, else `0`).

This is intentionally chunk-level because your target is: *"are base chunks retrieved?"*


In [ ]:
from pathlib import Path
import json
import math
import os
import random

import chromadb
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CHROMA_PATH = PROJECT_ROOT / "backend" / "data" / "chroma_db"
BASE_COLLECTION_NAME = "estate_documents"
BASE_EMBEDDING_MODEL = "text-embedding-3-small"
LARGE_EMBEDDING_MODEL = "text-embedding-3-large"
LARGE_COLLECTION_NAME = "estate_documents_eval_embed3_large"
QUESTION_GEN_MODEL = "gpt-4o-mini"
QUESTIONS_PER_CHUNK = 5
TOP_K = 5
RERANK_TOP_N = 20
RANDOM_SEED = 42
MAX_CHUNKS_FOR_GT = None  # e.g. 20 for faster/cheaper iteration.
REGENERATE_GROUND_TRUTH = False
RUN_LARGE_EMBED_EXPERIMENT = False
FORCE_REBUILD_LARGE_COLLECTION = False
GROUND_TRUTH_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "ground_truth_chunk_questions.jsonl"

load_dotenv(PROJECT_ROOT / ".env", override=True)
if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma path: {CHROMA_PATH}")
print(f"Base collection: {BASE_COLLECTION_NAME}")
print(f"Ground truth file: {GROUND_TRUTH_PATH}")


In [ ]:
if not CHROMA_PATH.exists():
    raise FileNotFoundError(f"Chroma path does not exist: {CHROMA_PATH}")

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
base_collection = client.get_or_create_collection(name=BASE_COLLECTION_NAME)
openai_client = OpenAI()


def fetch_all_chunks(chroma_collection, batch_size: int = 200) -> list[dict]:
    total = chroma_collection.count()
    rows: list[dict] = []

    for offset in range(0, total, batch_size):
        batch = chroma_collection.get(
            include=["documents", "metadatas"],
            limit=batch_size,
            offset=offset,
        )
        ids = batch.get("ids", [])
        docs = batch.get("documents", [])
        metas = batch.get("metadatas", [])

        for idx, chunk_id in enumerate(ids):
            rows.append(
                {
                    "chunk_id": chunk_id,
                    "text": docs[idx] or "",
                    "metadata": metas[idx] or {},
                }
            )

    return rows


all_chunks = fetch_all_chunks(base_collection)
print(f"Loaded {len(all_chunks)} chunk(s) from base collection")

random.seed(RANDOM_SEED)
if MAX_CHUNKS_FOR_GT is None or MAX_CHUNKS_FOR_GT >= len(all_chunks):
    eval_chunks = list(all_chunks)
else:
    eval_chunks = random.sample(all_chunks, MAX_CHUNKS_FOR_GT)

print(f"Using {len(eval_chunks)} chunk(s) for ground-truth generation")


In [ ]:
import re


def save_jsonl(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def extract_chunk_anchors(snippet: str) -> list[str]:
    anchors: list[str] = []

    date_matches = re.findall(
        r"\b[0-3]?\d\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+(?:19|20)\d{2}\b",
        snippet,
        flags=re.IGNORECASE,
    )
    amount_matches = re.findall(r"\b(?:EUR|€)\s*[0-9][0-9.,]*\b", snippet, flags=re.IGNORECASE)
    percent_matches = re.findall(r"\b[0-9]+(?:[.,][0-9]+)?%\b", snippet)
    quoted_matches = re.findall(r""([^"]{3,60})"", snippet)
    name_matches = re.findall(
        r"\b[A-Z][a-z]+(?:-[A-Z][a-z]+)?(?:\s+[A-Z][a-z]+(?:-[A-Z][a-z]+)?)+\b",
        snippet,
    )

    for group in [date_matches, amount_matches, percent_matches, quoted_matches, name_matches]:
        for item in group:
            item = " ".join(item.split()).strip()
            if len(item) < 3:
                continue
            if item.lower() in {"the borrowers", "the sellers", "the buyers", "the lender"}:
                continue
            if item not in anchors:
                anchors.append(item)

    return anchors[:20]


def is_chunk_specific_question(question: str, snippet: str, anchors: list[str]) -> bool:
    q = " ".join(question.strip().split())
    if len(q) < 12:
        return False

    q_lower = q.lower()
    banned_phrases = [
        "purpose of the document",
        "purpose of this document",
        "in the document",
        "from the document",
        "in this chunk",
        "in the chunk",
        "from this chunk",
        "from the passage",
        "main idea",
        "main purpose",
        "summarize",
    ]
    if any(phrase in q_lower for phrase in banned_phrases):
        return False

    if "document" in q_lower or "chunk" in q_lower or "passage" in q_lower:
        return False

    if any(anchor.lower() in q_lower for anchor in anchors):
        return True

    snippet_numbers = set(re.findall(r"\d+[.,]?\d*", snippet))
    question_numbers = set(re.findall(r"\d+[.,]?\d*", q))
    if snippet_numbers.intersection(question_numbers):
        return True

    return False


def generate_questions_for_chunk(chunk_text: str, n_questions: int = QUESTIONS_PER_CHUNK) -> list[str]:
    snippet = chunk_text.strip()[:2500]
    anchors = extract_chunk_anchors(snippet)
    anchor_line = ", ".join(f'"{a}"' for a in anchors[:12]) if anchors else "(none found)"

    system_prompt = (
        "You create chunk-grounded retrieval questions. "
        "Every question must target concrete facts in the chunk, not document-level summaries."
    )
    user_prompt = "\n".join(
        [
            f"Generate {n_questions} diverse user-style questions based ONLY on this chunk.",
            "Hard rules:",
            "- Each question must include at least one concrete anchor copied from the chunk (name/date/amount/term).",
            "- Do NOT use words: document, chunk, passage.",
            "- Do NOT ask broad summary/purpose questions.",
            "- Avoid yes/no questions.",
            "- Keep questions factual and specific.",
            "- Avoid copying full sentences from the chunk.",
            f"Suggested anchors: {anchor_line}",
            "Return strict JSON with schema:",
            '{"questions": ["...", "...", ...]}',
            "",
            "Chunk:",
            "---",
            snippet,
            "---",
        ]
    )

    for _ in range(5):
        response = openai_client.chat.completions.create(
            model=QUESTION_GEN_MODEL,
            temperature=0.1,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw = response.choices[0].message.content or "{}"

        try:
            parsed = json.loads(raw)
            candidates = parsed.get("questions", [])
            cleaned = []
            seen = set()
            for q in candidates:
                if not isinstance(q, str):
                    continue
                q = q.strip()
                if not q or q in seen:
                    continue
                if not is_chunk_specific_question(q, snippet=snippet, anchors=anchors):
                    continue
                seen.add(q)
                cleaned.append(q)
            if len(cleaned) >= n_questions:
                return cleaned[:n_questions]
        except Exception:
            pass

    raise RuntimeError("Could not generate sufficiently chunk-specific questions after 5 attempts.")


def build_ground_truth(chunks: list[dict], regenerate: bool = False) -> list[dict]:
    if GROUND_TRUTH_PATH.exists() and not regenerate:
        rows = load_jsonl(GROUND_TRUTH_PATH)
        print(f"Loaded existing ground truth: {len(rows)} question(s)")
        return rows

    rows: list[dict] = []
    total = len(chunks)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_id = chunk["chunk_id"]
        chunk_text = chunk["text"]
        metadata = chunk.get("metadata", {})

        questions = generate_questions_for_chunk(chunk_text, n_questions=QUESTIONS_PER_CHUNK)
        for q_idx, question in enumerate(questions, start=1):
            rows.append(
                {
                    "question_id": f"{chunk_id}-q{q_idx:02d}",
                    "question": question,
                    "expected_chunk_id": chunk_id,
                    "expected_document_id": metadata.get("document_id"),
                    "document_type": metadata.get("document_type"),
                    "page_number": metadata.get("page_number"),
                }
            )

        if idx % 10 == 0 or idx == total:
            print(f"Generated questions for {idx}/{total} chunk(s)")

    save_jsonl(rows, GROUND_TRUTH_PATH)
    print(f"Saved ground truth: {len(rows)} question(s) -> {GROUND_TRUTH_PATH}")
    return rows


ground_truth_rows = build_ground_truth(
    eval_chunks,
    regenerate=REGENERATE_GROUND_TRUTH,
)
print(f"Ground-truth sample:\n{ground_truth_rows[0] if ground_truth_rows else 'No rows'}")


In [ ]:
def embed_texts(texts: list[str], model: str) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model=model,
        input=texts,
    )
    return [item.embedding for item in response.data]


def embed_query(query: str, model: str) -> list[float]:
    return embed_texts([query], model=model)[0]


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def ensure_large_embedding_collection(
    source_collection_name: str,
    target_collection_name: str,
    model_name: str,
    force_rebuild: bool = False,
    batch_size: int = 64,
):
    source = client.get_or_create_collection(name=source_collection_name)

    if force_rebuild:
        try:
            client.delete_collection(name=target_collection_name)
        except Exception:
            pass

    target = client.get_or_create_collection(name=target_collection_name)

    if not force_rebuild and target.count() == source.count() and target.count() > 0:
        print(f"Using existing large collection '{target_collection_name}' ({target.count()} vectors)")
        return target

    if target.count() > 0:
        client.delete_collection(name=target_collection_name)
        target = client.get_or_create_collection(name=target_collection_name)

    rows = fetch_all_chunks(source)
    total = len(rows)

    for start in range(0, total, batch_size):
        batch = rows[start : start + batch_size]
        ids = [row["chunk_id"] for row in batch]
        docs = [row["text"] for row in batch]
        metas = [row["metadata"] for row in batch]
        embeds = embed_texts(docs, model=model_name)
        target.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=embeds)

        if (start // batch_size + 1) % 5 == 0 or start + len(batch) == total:
            print(f"Built large collection vectors: {start + len(batch)}/{total}")

    return target


def retrieve_candidates(
    query: str,
    collection_name: str,
    query_model: str,
    top_n: int,
    where: dict | None = None,
) -> list[dict]:
    collection = client.get_or_create_collection(name=collection_name)
    query_embedding = embed_query(query, model=query_model)

    query_kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": top_n,
        "include": ["documents", "metadatas", "distances"],
    }
    if where is not None:
        query_kwargs["where"] = where

    result = collection.query(**query_kwargs)
    ids = result.get("ids", [[]])[0]
    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]
    dists = result.get("distances", [[]])[0]

    candidates = []
    for i, chunk_id in enumerate(ids):
        candidates.append(
            {
                "id": chunk_id,
                "document": docs[i] or "",
                "metadata": metas[i] or {},
                "distance": dists[i],
            }
        )
    return candidates


DOC_EMBED_CACHE: dict[tuple[str, str], list[float]] = {}


def rerank_candidates_with_embeddings(
    query: str,
    candidates: list[dict],
    rerank_model: str,
) -> list[dict]:
    if not candidates:
        return candidates

    query_embedding = embed_query(query, model=rerank_model)

    missing_ids = []
    missing_docs = []
    for cand in candidates:
        key = (rerank_model, cand["id"])
        if key not in DOC_EMBED_CACHE:
            missing_ids.append(cand["id"])
            missing_docs.append(cand["document"])

    if missing_docs:
        embeddings = embed_texts(missing_docs, model=rerank_model)
        for cid, emb in zip(missing_ids, embeddings):
            DOC_EMBED_CACHE[(rerank_model, cid)] = emb

    reranked = []
    for cand in candidates:
        emb = DOC_EMBED_CACHE[(rerank_model, cand["id"])]
        score = cosine_similarity(query_embedding, emb)
        reranked.append({**cand, "rerank_score": score})

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked


def reciprocal_rank(retrieved_ids: list[str], expected_id: str) -> tuple[float, int | None]:
    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id == expected_id:
            return 1.0 / rank, rank
    return 0.0, None


def evaluate_retrieval(
    ground_truth: list[dict],
    *,
    collection_name: str,
    query_model: str,
    top_k: int = TOP_K,
    first_stage_top_n: int | None = None,
    rerank_model: str | None = None,
    where: dict | None = None,
):
    rows: list[dict] = []
    top_n = max(top_k, first_stage_top_n or top_k)

    for idx, item in enumerate(ground_truth, start=1):
        candidates = retrieve_candidates(
            item["question"],
            collection_name=collection_name,
            query_model=query_model,
            top_n=top_n,
            where=where,
        )

        if rerank_model is not None:
            candidates = rerank_candidates_with_embeddings(
                item["question"],
                candidates,
                rerank_model=rerank_model,
            )

        retrieved_ids = [cand["id"] for cand in candidates[:top_k]]
        rr, rank = reciprocal_rank(retrieved_ids, item["expected_chunk_id"])
        hit = int(rank is not None)

        rows.append(
            {
                **item,
                "retrieved_ids": retrieved_ids,
                "hit": hit,
                "rank": rank,
                "reciprocal_rank": rr,
            }
        )

        if idx % 50 == 0 or idx == len(ground_truth):
            print(f"Evaluated {idx}/{len(ground_truth)} question(s)")

    n = len(rows) or 1
    hit_rate = sum(row["hit"] for row in rows) / n
    mrr = sum(row["reciprocal_rank"] for row in rows) / n

    summary = {
        "num_questions": len(rows),
        "top_k": top_k,
        "first_stage_top_n": top_n,
        "collection_name": collection_name,
        "query_model": query_model,
        "rerank_model": rerank_model,
        "hit_rate": hit_rate,
        "mrr": mrr,
    }
    return summary, rows


In [ ]:
# Optional filter example:
# WHERE_FILTER = {"document_type": "mortgage_deed"}
WHERE_FILTER = None

experiments = [
    {
        "name": "baseline_small",
        "collection_name": BASE_COLLECTION_NAME,
        "query_model": BASE_EMBEDDING_MODEL,
        "first_stage_top_n": TOP_K,
        "rerank_model": None,
    },
    {
        "name": "small_plus_large_rerank",
        "collection_name": BASE_COLLECTION_NAME,
        "query_model": BASE_EMBEDDING_MODEL,
        "first_stage_top_n": RERANK_TOP_N,
        "rerank_model": LARGE_EMBEDDING_MODEL,
    },
]

if RUN_LARGE_EMBED_EXPERIMENT:
    ensure_large_embedding_collection(
        source_collection_name=BASE_COLLECTION_NAME,
        target_collection_name=LARGE_COLLECTION_NAME,
        model_name=LARGE_EMBEDDING_MODEL,
        force_rebuild=FORCE_REBUILD_LARGE_COLLECTION,
    )
    experiments.append(
        {
            "name": "large_index",
            "collection_name": LARGE_COLLECTION_NAME,
            "query_model": LARGE_EMBEDDING_MODEL,
            "first_stage_top_n": TOP_K,
            "rerank_model": None,
        }
    )

results_by_experiment = {}
for exp in experiments:
    print("=" * 80)
    print(f"Running experiment: {exp['name']}")

    summary, eval_rows = evaluate_retrieval(
        ground_truth_rows,
        collection_name=exp["collection_name"],
        query_model=exp["query_model"],
        top_k=TOP_K,
        first_stage_top_n=exp["first_stage_top_n"],
        rerank_model=exp["rerank_model"],
        where=WHERE_FILTER,
    )

    results_by_experiment[exp["name"]] = {
        "summary": summary,
        "rows": eval_rows,
    }

    print(f"hit_rate={summary['hit_rate']:.4f} | mrr={summary['mrr']:.4f}")

print("
Experiment summary")
for name, payload in results_by_experiment.items():
    s = payload["summary"]
    print(f"- {name}: hit_rate={s['hit_rate']:.4f}, mrr={s['mrr']:.4f}")


In [ ]:
PRIMARY_EXPERIMENT = "small_plus_large_rerank"
if PRIMARY_EXPERIMENT not in results_by_experiment:
    PRIMARY_EXPERIMENT = next(iter(results_by_experiment.keys()))

primary_rows = results_by_experiment[PRIMARY_EXPERIMENT]["rows"]
misses = [row for row in primary_rows if row["hit"] == 0]
print(f"Experiment: {PRIMARY_EXPERIMENT}")
print(f"Misses: {len(misses)} / {len(primary_rows)}")

for row in misses[:5]:
    print("=" * 100)
    print(f"Question: {row['question']}")
    print(f"Expected chunk: {row['expected_chunk_id']}")
    print(f"Top-{TOP_K} retrieved: {row['retrieved_ids']}")
